# WMDP Canonical Script Mini GPU Test

This notebook validates the canonical `llm-unlearn-eco/scripts/evaluate_wmdp.py` entrypoint from a Kaggle-style clean clone. It runs `sample_size=2` for no-corrupt and classifier-free corrupt-hook modes.

Defaults: `PORT_TARGET_CONFIG_NAME=phi-1_5`, `PORT_ATTN_IMPLEMENTATION=eager`, `PORT_TORCH_DTYPE=float16`, `PORT_WMDP_BATCH_SIZE=1`.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This smoke test is intended for Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')

## Run Canonical Script

In [ ]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH')
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = int(os.environ.get('PORT_WMDP_SAMPLE_SIZE', '2'))
RUN_ORIGINAL = os.environ.get('PORT_RUN_ORIGINAL', '1').strip().lower() not in {'0', 'false', 'no'}
RUN_CORRUPT = os.environ.get('PORT_RUN_CORRUPT', '1').strip().lower() not in {'0', 'false', 'no'}

SCRIPT_PATH = PROJECT_ROOT / 'llm-unlearn-eco' / 'scripts' / 'evaluate_wmdp.py'
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'

def base_command(run_name, task_config):
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        '--model_name', TARGET_CONFIG_NAME,
        '--task_config', task_config,
        '--sample_size', str(SAMPLE_SIZE),
        '--batch_size', str(BATCH_SIZE),
        '--torch_dtype', TORCH_DTYPE,
        '--attn_implementation', ATTN_IMPLEMENTATION,
        '--output_dir', str(OUTPUT_DIR),
        '--run_name', run_name,
    ]
    if TARGET_HF_NAME:
        cmd.extend(['--target_hf_name', TARGET_HF_NAME])
    if TARGET_MODEL_PATH:
        cmd.extend(['--model_path', TARGET_MODEL_PATH])
    return cmd

commands = []
if RUN_ORIGINAL:
    commands.append(base_command(
        f'wmdp_script_mini_original_{TARGET_CONFIG_NAME}',
        'multiple_choice_original.yaml',
    ))
if RUN_CORRUPT:
    cmd = base_command(
        f'wmdp_script_mini_zero_out_{TARGET_CONFIG_NAME}',
        'multiple_choice_zero_out.yaml',
    )
    cmd.append('--attack_all_prompts')
    commands.append(cmd)

for cmd in commands:
    print('\n$ ' + ' '.join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))

## Validate Artifacts

In [ ]:
import pandas as pd

expected_rows = SAMPLE_SIZE * 3
run_names = []
if RUN_ORIGINAL:
    run_names.append(f'wmdp_script_mini_original_{TARGET_CONFIG_NAME}')
if RUN_CORRUPT:
    run_names.append(f'wmdp_script_mini_zero_out_{TARGET_CONFIG_NAME}')

for run_name in run_names:
    run_dir = OUTPUT_DIR / run_name
    summary_path = run_dir / 'summary.json'
    predictions_path = run_dir / 'predictions.csv'
    summary_by_run_path = run_dir / 'summary_by_run.csv'
    for path in [summary_path, predictions_path, summary_by_run_path, run_dir / 'run_config.json']:
        if not path.exists():
            raise FileNotFoundError(path)
    predictions = pd.read_csv(predictions_path)
    if len(predictions) != expected_rows:
        raise AssertionError(f'{run_name}: expected {expected_rows} prediction rows, got {len(predictions)}')
    summary = json.loads(summary_path.read_text())
    print('\n', run_name)
    print('rows:', len(predictions))
    print(pd.read_csv(summary_by_run_path))
    print('summary_overall:', summary.get('summary_overall'))

print('WMDP CANONICAL SCRIPT MINI GPU TEST COMPLETED')